Question 1) List two benefits of using Mongoose instead of the native MongoDB driver. Also mention one scenario where using the native driver could be preferable.

Answer:

Benefits of Using Mongoose

- Schema Validation and Data Modeling: Mongoose allows us to define a strict structure (Schema) for your documents, ensuring that data follows specific types and required fields before it ever reaches the database.


- Built-in Middleware (Hooks): It provides "pre" and "post" hooks, allowing us to automatically trigger functions (like hashing a password or logging) before or after operations like save() or find().


- Object-Document Mapping (ODM): Mongoose maps MongoDB documents to rich JavaScript objects with built-in methods, making the code more readable and easier to manage for complex applications

Scenario for Using the Native MongoDB Driver

High-Performance/Bulk Operations: The native driver is preferable when we need maximum performance with minimal overhead, such as during massive data migrations or high-frequency ingestion where the abstraction layer of Mongoose (validation, hooks, and casting) might slow down the process.

Question 2) Create a Mongoose schema for a `User` with properties:  
- `name` (required string, minimum length 2, maximum length 50)  
- `email` (required string)  

Then export a `User` model.

Answer :



```
import mongoose from 'mongoose';

const userSchema = new mongoose.Schema({
  name: {
    type: String,
    required: [true, 'Name is required']
    minlength: [2, 'Name must be at least 2 characters'],
    maxlength: [50, 'Name cannot exceed 50 characters'],
    trim: true
  },
  email: {
    type: String,
    required: [true, 'Email is required'],
    unique: true,
    lowercase: true,
    trim: true
  }
}, {
  timestamps: true
});

// Create and export the User model
const User = mongoose.model('User', userSchema);
export default User;
```



Question 3) Write code to (a) fetch all users, (b) fetch a user by ID, and (c) add a query helper byEmailDomain(domain) that returns users whose email ends with the given domain. Show example usage.

Answer:


```
import User from './models/User.js';

userSchema.query.byEmailDomain = function(domain) {
  return this.where({ email: new RegExp(`@${domain}$`, 'i') });
};

// (a) Fetch all users
const getAllUsers = async () => {
  return await User.find();
};

// (b) Fetch a user by ID
const getUserById = async (id) => {
  return await User.findById(id);
};
```
Example Usage



```
async function run() {
  try {
    // 1. Fetching all users
    const allUsers = await getAllUsers();
    console.log('All Users:', allUsers);

    // 2. Fetching by ID (example ID)
    const singleUser = await getUserById('6543210abcdef1234567890a');
    console.log('Single User:', singleUser);

    // 3. Using the Query Helper
    const gmailUsers = await User.find().byEmailDomain('gmail.com');
    console.log('Gmail Users:', gmailUsers);

  } catch (error) {
    console.error('Error fetching users:', error);
  }
}
```




Question 4) Demonstrate two ways to change a user’s name to "Rita" using Mongoose: (1) load, modify, then save(), and (2) use findOneAndUpdate(). Mention when save() is preferable.

Answer:

1st WAY : Load, Modify, then save()


```
const updateWithSave = async (userId) => {
  // Load the document
  const user = await User.findById(userId);
  
  if (user) {
    // Modify the property
    user.name = "Rita";
    
    // Save the changes
    await user.save();
    console.log("User updated successfully via save()");
  }
};

```
2ND WAY : Using findOneAndUpdate()



```
const updateWithFindOneAndUpdate = async (userId) => {
  const updatedUser = await User.findOneAndUpdate(
    { _id: userId },
    { name: "Rita" },
    { new: true }
  );
  console.log("User updated via findOneAndUpdate()");
};
```



Question 5) Explain when you would embed a subdocument versus using references in MongoDB/Mongoose. Show code for a one-to-many reference (e.g. a Post having many Comment references).

Answer:


Choosing between embedding and referencing is a key decision in MongoDB data modeling that impacts performance and scalability.

When to use Embedding (Subdocuments) - Use for "one-to-few" relationships where the data is small and tightly coupled.

When to use Referencing (Normalization) - Use for "one-to-many" or "many-to-many" where the data grows significantly

Code Example: One-to-Many Reference

Comment Model



```
import mongoose from 'mongoose';

const commentSchema = new mongoose.Schema({
  text: { type: String, required: true },
  author: String,
  date: { type: Date, default: Date.now }
});

const Comment = mongoose.model('Comment', commentSchema);
export default Comment;
```

Post Model (with References)


```
const postSchema = new mongoose.Schema({
  title: { type: String, required: true },
  content: String,

  comments: [{
    type: mongoose.Schema.Types.ObjectId,
    ref: 'Comment'
  }]
});

const Post = mongoose.model('Post', postSchema);
export default Post;
```

Fetching Data (Populating)



```
const getPostWithComments = async (postId) => {
  return await Post.findById(postId).populate('comments');
};
```






Question 6) Write a Mongoose (or MongoDB) aggregation pipeline to group users by email domain and count how many users per domain. Then show how you could use $lookup to join a bounces collection (with bounce counts per domain).

Answer:

Grouping Users by Email Domain

```
const domainCountPipeline = [
  {
    
    $project: {
      domain: { $arrayElemAt: [{ $split: ["$email", "@"] }, 1] }
    }
  },
  {
    $group: {
      _id: "$domain",
      userCount: { $sum: 1 }
    }
  }
];

const results = await User.aggregate(domainCountPipeline);
```

Using $lookup to Join Bounces Collection



```
const joinBouncesPipeline = [
  {
    $project: {
      domain: { $arrayElemAt: [{ $split: ["$email", "@"] }, 1] }
    }
  },
  {
    $group: {
      _id: "$domain",
      userCount: { $sum: 1 }
    }
  },
  {
    $lookup: {
      from: "bounces",         
      localField: "_id",        
      foreignField: "domain",   
      as: "bounceDetails"       
    }
  }
];

const fullReport = await User.aggregate(joinBouncesPipeline);
```



Question 7) Show minimal code (ESM) to connect to MongoDB Atlas using Mongoose. Assume the URI is stored in the environment variable MONGODB_URI.

Answer:


```
import mongoose from 'mongoose';


const connectDB = async () => {
  try {
    const conn = await mongoose.connect(process.env.MONGODB_URI);
    console.log(`MongoDB Connected: ${conn.connection.host}`);
  } catch (error) {
    console.error(`Error: ${error.message}`);
    process.exit(1);
  }
};

export default connectDB;
```




Question 8) Give a short description of Express. Then write routes for:
●	GET /health → responds “OK”
●	GET /users/:id → responds with { id: <id> }
●	GET /search?term=... → responds with { term: <term> }

Answer :



```
import express from 'express';

const app = express();

app.get('/health', (req, res) => {
  res.send('OK');
});

app.get('/users/:id', (req, res) => {
  const { id } = req.params;
  res.json({ id: id });
});

app.get('/search', (req, res) => {
  const { term } = req.query;
  res.json({ term: term });
});

const PORT = 3000;
app.listen(PORT, () => console.log(`Server running on port ${PORT}`));
```



Question 9) Create a router for /api/users with two endpoints: GET / returning empty array and POST / returning the posted JSON. Also add a logger middleware that prints method and URL for every request.

Answer:

Logger Middleware



```
const logger = (req, res, next) => {
  console.log(`${req.method} ${req.url}`);
  next();
};
```

User Router (/api/users)


```
import express from 'express';
const router = express.Router();

router.get('/', (req, res) => {
  res.json([]);
});


router.post('/', (req, res) => {
  res.json(req.body);
});

export default router;
```
Main Application Setup



```
import express from 'express';
import userRouter from './routes/userRouter.js';

const app = express();

app.use(express.json());

app.use(logger);

app.use('/api/users', userRouter);

app.listen(3000, () => console.log('Server running on port 3000'));
```






Question 10) Add an error-handling middleware to catch thrown errors and respond with { error: <message> }, status 500. Demonstrate by a route that throws an error.

Answer:

Error-Handling Middleware

This middleware catches any error passed down the line (via throw or next(err)) and sends a standardized JSON response with a 500 status code.


```
const errorHandler = (err, req, res, next) => {
  console.error(err.stack);
  
  res.status(500).json({
    error: err.message || "Internal Server Error"
  });
};
```
Demonstration Route



```
import express from 'express';
const app = express();


app.get('/trigger-error', (req, res) => {
  throw new Error("Something went wrong!");
});

app.use(errorHandler);

app.listen(3000, () => console.log('Server running on port 3000'));
```





Question 11) Write middleware auth() that checks for Authorization: Bearer <token> header, verifies the token, and sets req.user. Then protect route GET /me to return user info if valid, otherwise 401.

Answer:
Authentication Middleware (auth)

This middleware checks the Authorization header for the "Bearer" scheme, simulates token verification, and attaches the user data to the req object.


```
import jwt from 'jsonwebtoken';


const auth = (req, res, next) => {
  const authHeader = req.headers.authorization;

  if (!authHeader || !authHeader.startsWith('Bearer ')) {
    return res.status(401).json({ error: 'Access denied. No token provided.' });
  }

  const token = authHeader.split(' ')[1];

  try {

    const decoded = jwt.verify(token, process.env.JWT_SECRET || 'your_secret_key');
    
    req.user = decoded;
    next();
  } catch (error) {

    res.status(401).json({ error: 'Invalid or expired token.' });
  }
};
```

Protected Route (GET /me)



```
import express from 'express';
const app = express();

app.get('/me', auth, (req, res) => {

  res.json({
    message: "User info retrieved successfully",
    user: req.user
  });
});

app.listen(3000, () => console.log('Server running on port 3000'));
```




